In [14]:
!pip install -q google-generativeai faiss-cpu numpy

import numpy as np
import faiss
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
chat_model = genai.GenerativeModel('gemini-3.6-flash')
print("Setup done.")

Setup done.


In [15]:
knowledge_base = [
    "Nimbus Robotics was founded in 2031 by engineer Priya Kalathil in Pune, India.",
    "Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a 99.2% accuracy rate.",
    "The Aster-7 uses a proprietary gripper called FlexGrip, which adjusts pressure using 12 micro-sensors per finger.",
    "Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033.",
    "Nimbus Robotics' main competitor is Solace Automation, founded a year earlier in 2030.",
    "The Aster-7's battery lasts 14 hours on a single charge and recharges fully in 40 minutes.",
    "Nimbus Robotics employs 212 people across three offices: Pune, Bengaluru, and Singapore.",
    "The company's CTO, Rohan Mehta, previously led robotics research at a university lab for eight years.",
    "Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035.",
    "The Aster-7 has been deployed in over 60 warehouses across South and Southeast Asia."
]

def get_embeddings(texts, model="models/gemini-embedding-001"):
    embeddings = []
    for text in texts:
        result = genai.embed_content(model=model, content=text)
        embeddings.append(result['embedding'])
    return embeddings

corpus_embeddings = get_embeddings(knowledge_base)
corpus_vectors = np.array(corpus_embeddings, dtype=np.float32)

dimension = corpus_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_vectors)
print(f"FAISS index size: {index.ntotal} vectors")

FAISS index size: 10 vectors


In [16]:
def retrieve_chunks(query, top_k=3):
    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype=np.float32)
    distances, indices = index.search(query_vector, top_k)
    return [knowledge_base[idx] for idx in indices[0]]

In [17]:
def generate_with_rag(query):
    chunks = retrieve_chunks(query, top_k=3)
    context = "\n".join(f"- {c}" for c in chunks)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the context doesn't contain the answer, say "I don't have that information."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    response = chat_model.generate_content(prompt)
    return response.text, chunks

def generate_without_rag(query):
    prompt = f"Answer this question: {query}"
    response = chat_model.generate_content(prompt)
    return response.text

In [19]:
import time

for q in test_queries:
    print("=" * 70)
    print(f"QUERY: {q}\n")

    rag_answer, used_chunks = generate_with_rag(q)
    time.sleep(15)  # stay under free-tier rate limit
    no_rag_answer = generate_without_rag(q)
    time.sleep(15)

    print("WITHOUT RAG (no context):")
    print(f"  {no_rag_answer}\n")

    print("WITH RAG (grounded):")
    print(f"  {rag_answer}")
    print(f"  Retrieved chunks used:")
    for c in used_chunks:
        print(f"    - {c}")
    print()

QUERY: Who founded Nimbus Robotics and when?

WITHOUT RAG (no context):
  Because "Nimbus Robotics" can refer to a few different organizations depending on the context, here are the primary real-world entities associated with that name:

1. **NIMBUS Lab (Nebraska Intelligent MoBiLe Unmanned Systems Lab)**
   * **Founders:** **Carrick Detweiler** and **Sebastian Elbaum**
   * **When:** **2010**
   * **Details:** Located at the University of Nebraska–Lincoln, this is a prominent robotics research lab focused on software, hardware, and interactions of unmanned aerial vehicles (UAVs) and autonomous systems.

2. **Nimbus S.r.l. (Italian Drone/Robotics Manufacturer)**
   * **Founders:** Founded by **Giuseppe Capuano** and a team of aerospace engineers.
   * **When:** **2006**
   * **Details:** An Italian technology company that designs and manufactures unmanned aerial vehicles (UAVs) and robotic systems for civilian and security applications.

*If you are referring to a specific fictional co

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 11639.40ms


WITHOUT RAG (no context):
  As of current information, there is **no widely recognized tech product, missile system, or major release officially named "Aster-8"** with an upcoming scheduled release date. 

However, depending on what you might be referring to, here is the status of a few possibilities:

1. **ASTER Multiseat Software (by IBIK):**
   * If you are referring to the PC multiseat software **ASTER V8**, it has already been released and is currently available for Windows. 

2. **MBDA Aster Missile Family (Defense):**
   * The French/Italian Aster missile family includes the **Aster 15** and **Aster 30**. The latest upgraded variant, the **Aster 30 Block 1 NT** (New Technology), is scheduled for operational deployment starting around **2024–2025**. There is no "Aster-8" variant.

3. **Video Games / Sci-Fi / Tech Codenames:**
   * "Aster" has been used as an internal codename in the past (such as the 2014 HTC Desire Eye smartphone, codenamed *HTC Aster*). 

If you are referring t